# CTRL-MATH AIMO3 — Gateway Submission Notebook

**Competition**: AI Mathematical Olympiad Progress Prize 3 (AIMO3)

This notebook integrates with the `kaggle_evaluation` gRPC gateway pattern.
The gateway sends problems one at a time via `predict(df)` where `df` is a
polars DataFrame with columns `id` and `problem`.

**Pipeline**:
1. Install offline dependencies
2. Add h1-main code directory to sys.path
3. Initialize all CTRL-MATH modules (safe imports)
4. Load LLM (Qwen2.5-Math-14B-Instruct + optional LoRA)
5. Create SolveOrchestrator
6. Define `predict(df)` function for the gateway
7. Start InferenceServer

In [ ]:
# ── Cell 0: Install dependencies ────────────────────────────────────────────
import subprocess, sys, os

# Kaggle offline mode: install from pre-downloaded wheels
WHEELS_DIR = "/kaggle/input/ctrlmath-wheels"
if os.path.isdir(WHEELS_DIR):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-index",
        "--find-links", WHEELS_DIR, "-q",
        "numpy", "numba", "sympy", "torch", "transformers",
        "accelerate", "bitsandbytes", "peft", "trl", "z3-solver",
        "sentence-transformers", "scipy", "polars",
    ])
    print("\u2705 Offline packages installed from wheels.")
else:
    print("\u26a0\ufe0f No offline wheels found. Using pre-installed packages.")

# Add h1-main code directory to sys.path
CODE_DIR = "/kaggle/input/h1-main"
if os.path.isdir(CODE_DIR) and CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
    print(f"\u2705 Added {CODE_DIR} to sys.path")

# Set Numba cache directory
os.environ["NUMBA_CACHE_DIR"] = "/kaggle/working/.numba_cache"
os.makedirs("/kaggle/working/.numba_cache", exist_ok=True)
print("\u2705 Environment configured.")

In [ ]:
# ── Cell 1: Load all CTRL-MATH modules (safe imports) ───────────────────────
import time
t0 = time.time()

# Core number theory (Numba JIT)
try:
    from cell_02a_numba_nt import (
        vp_jit, vp_factorial_jit, vp_binomial_jit,
        dirichlet_conv_safe, powmod_batch, modinv_batch,
        fib_jit, sigma_k_sieve, poly_mul_ntt, ntt,
    )
    print("\u2705 Numba NT loaded")
except Exception as e:
    print(f"\u26a0\ufe0f Numba NT unavailable: {e}")

# GPU acceleration
try:
    from cell_02b_cupy_gpu import GPUArithmetic
    print("\u2705 CuPy GPU loaded")
except Exception as e:
    print(f"\u26a0\ufe0f CuPy GPU unavailable: {e}")

# Problem parsing
try:
    from cell_03_mog_parser import MOGParser
    print("\u2705 MOG parser loaded")
except Exception as e:
    print(f"\u26a0\ufe0f MOG parser unavailable: {e}")

# Domain solvers
try:
    from cell_04_transform_engine import TransformEngine, TransformResult
    print("\u2705 Transform engine loaded")
except Exception as e:
    print(f"\u26a0\ufe0f Transform engine unavailable: {e}")

try:
    from cell_04b_linear_recurrence import LinearRecurrenceSolver
    from cell_04c_combinatorics import CombinatoricsSolver
    from cell_04d_number_theory import NumberTheorySolver
    from cell_04e_gf_solver import GFSolver
    from cell_04f_geometry import GeometrySolver
    from cell_04g_geometry_prover import geometry_tool, GeometryTool
    print("\u2705 Domain solvers loaded")
except Exception as e:
    print(f"\u26a0\ufe0f Some domain solvers unavailable: {e}")

# MCTS + MPC
try:
    from cell_06_mcts import MCTSEngine
    from cell_06_mpc_planner import MPCPlanner, solve_aimo3
    print("\u2705 MCTS/MPC loaded")
except Exception as e:
    print(f"\u26a0\ufe0f MCTS/MPC unavailable: {e}")

# LLM executor
try:
    from cell_07_llm_executor_v5 import LLMExecutorV5
    print("\u2705 LLM executor loaded")
except Exception as e:
    LLMExecutorV5 = None
    print(f"\u26a0\ufe0f LLM executor unavailable: {e}")

# PRM + Kalman
try:
    from cell_08_prm import ProcessRewardModel
    from cell_08_kalman import KalmanBeliefState
    print("\u2705 PRM/Kalman loaded")
except Exception as e:
    print(f"\u26a0\ufe0f PRM/Kalman unavailable: {e}")

# Verification
try:
    from cell_09_mathrag import MathRAG
    print("\u2705 MathRAG loaded")
except Exception as e:
    print(f"\u26a0\ufe0f MathRAG unavailable: {e}")

try:
    from cell_09b_z3_parallel import ParallelZ3Checker
    print("\u2705 Z3 parallel loaded")
except Exception as e:
    ParallelZ3Checker = None
    print(f"\u26a0\ufe0f Z3 parallel unavailable: {e}")

try:
    from cell_10_template_store import TemplateStore
    from cell_11_answer_extractor import AnswerExtractor
    print("\u2705 Template store/Answer extractor loaded")
except Exception as e:
    print(f"\u26a0\ufe0f Template/Extractor unavailable: {e}")

# Orchestration
try:
    from cell_12_time_allocator import TimeAllocator
    from cell_13_self_consistency import SelfConsistencyChecker
    from cell_14_verification_ladder import VerificationLadder
    from cell_15_orchestrator_v5 import SolveOrchestrator
    print("\u2705 Orchestration modules loaded")
except Exception as e:
    print(f"\u26a0\ufe0f Orchestration modules unavailable: {e}")

print(f"\n\u2705 All CTRL-MATH modules loaded in {time.time() - t0:.1f}s")

In [ ]:
# ── Cell 2: Initialize LLM (Qwen2.5-Math-14B + optional LoRA) ───────────────
import torch
import os

# Model paths (Kaggle datasets)
PRIMARY_MODEL = "/kaggle/input/qwen2.5-math-14b-instruct"
ENSEMBLE_MODEL = "/kaggle/input/deepseek-math-7b-instruct"
LORA_ADAPTER = "/kaggle/working/ctrlmath_aimo3_lora"

# Fallback to HuggingFace Hub if not in Kaggle
if not os.path.isdir(PRIMARY_MODEL):
    PRIMARY_MODEL = "Qwen/Qwen2.5-Math-14B-Instruct"
if not os.path.isdir(ENSEMBLE_MODEL):
    ENSEMBLE_MODEL = "deepseek-ai/DeepSeek-Math-7B-Instruct"
if not os.path.isdir(LORA_ADAPTER):
    LORA_ADAPTER = None

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("\u26a0\ufe0f No GPU available \u2014 will use CPU-only mode")

# Initialize LLM executor
llm = None
if LLMExecutorV5 is not None:
    try:
        llm = LLMExecutorV5(
            primary_model=PRIMARY_MODEL,
            ensemble_model=ENSEMBLE_MODEL,
            lora_adapter=LORA_ADAPTER,
            load_in_4bit=True,
            use_flash_attn=True,
        )
        print("\u2705 LLM initialized.")
    except Exception as e:
        print(f"\u26a0\ufe0f LLM loading failed: {e}")
        print("Using CPU-only mode (no LLM inference)")

# Initialize PRM
prm = None
try:
    prm = ProcessRewardModel()
    print("\u2705 PRM initialized.")
except Exception as e:
    print(f"\u26a0\ufe0f PRM unavailable: {e}")

# Initialize Z3 checker
z3_checker = None
if ParallelZ3Checker is not None:
    try:
        z3_checker = ParallelZ3Checker(n_workers=4)
        print("\u2705 Z3 checker initialized.")
    except Exception as e:
        print(f"\u26a0\ufe0f Z3 checker unavailable: {e}")

print("\u2705 Models initialized.")

In [ ]:
# ── Cell 3: Build orchestrator ───────────────────────────────────────────────

orchestrator = SolveOrchestrator(
    llm=llm,
    prm=prm,
    z3_checker=z3_checker,
    total_seconds=9 * 3600.0,  # 9-hour competition limit
    n_problems=50,
    mcts_sims=128,
)
print("\u2705 SolveOrchestrator ready.")

In [ ]:
# ── Cell 4: Define predict() function for AIMO3 gateway ─────────────────────
import polars as pl

def predict(df: pl.DataFrame) -> pl.DataFrame:
    """
    AIMO3 gateway predict function.

    The gateway calls predict() with a single-row polars DataFrame containing
    columns 'id' and 'problem'. Returns a polars DataFrame with column 'answer'.

    Answers are integers in [0, 99999] (mod 10^5).
    Never raises — catches all exceptions and returns 0.
    """
    try:
        problem_id   = str(df["id"][0])
        problem_text = str(df["problem"][0])
        answer = orchestrator.solve_problem(problem_id, problem_text)
        answer = int(answer) % 100_000
        answer = max(0, answer)
    except Exception:
        answer = 0
    return pl.DataFrame({"answer": [answer]})

print("\u2705 predict() function defined.")

In [ ]:
# ── Cell 5: Start InferenceServer ────────────────────────────────────────────
import kaggle_evaluation.aimo_3_inference_server as aimo3_server

# Determine whether running in competition rerun or local testing
IS_COMPETITION_RERUN = os.path.exists("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv")

server = aimo3_server.AIMO3InferenceServer(predict)

if IS_COMPETITION_RERUN:
    # Competition private rerun: serve via gRPC gateway
    print("\u2705 Starting gRPC inference server for competition rerun...")
    server.serve()
else:
    # Local testing: run local gateway with test.csv
    TEST_CSV_PATHS = [
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",
        "/kaggle/input/aimo-2025-progress-prize-3/test.csv",
        "test.csv",
    ]
    test_csv = next((p for p in TEST_CSV_PATHS if os.path.exists(p)), None)
    if test_csv:
        print(f"\u2705 Running local gateway with {test_csv}")
        server.run_local_gateway(test_csv)
    else:
        print("\u26a0\ufe0f No test.csv found. Running server.serve() for manual testing.")
        server.serve()